# Notebook 0: Static Properties of the Aggregation Model

This notebook studies Mitchell Berman's aggregation model with the principle
weights held fixed by hand. It does not ask where the weights come from; that
question is taken up in the later notebooks. The goal here is narrower: assuming
we are handed a sensible set of weights, does the model behave the way Berman
says it should?

In Berman's model each legal principle has a **weight** (its general importance
in the system, a number in [0, 1]) and, in a particular case, an **activation**
(how strongly and in which direction it bears on the outcome, a number in
[-1, 1]). The net legal force on the question is the weighted sum

$$F = \sum_i w_i \, a_i.$$

The case is **determinate** when $|F| > \tau$ for a threshold $\tau$, and the
winning side is the sign of $F$.

We test four claims Berman makes about this model, plus one descriptive check.
The labels R2, R3, R4 are the requirement names used across the project (R1,
weight stability, applies only once weights are *learned*, so it is tested in
the later notebooks).

- **R2 — Rough weights suffice.** Berman argues (replying to Stephen Perry) that
  the model stays determinate even when the weights are only approximately right.
  We measure how much random error a weight absorbs before the verdict changes.
- **R3 — Determination is nonlexical.** This is Berman's central claim. The
  outcome depends on the *magnitudes* of the activations added together, not just
  on which single principle ranks highest. We measure whether the verdict
  responds to changes in magnitude that leave every sign and rank fixed.
- **R4 — More determinate law than Hart.** We compare the model's determinacy
  rate against Hart-style models that require official consensus.
- **No contradiction.** When principles conflict, a consensus model can validate
  both sides at once (a contradiction) or neither (a gap). A sum always returns
  one verdict. We show this directly.
- **D2 — Correlated activations (descriptive).** Real cases have correlated
  activations, which help the model. Reported for context, not used to judge the
  theory.

Every step is seeded, so the numbers reproduce exactly. The last cell re-checks
the headline results and stops with an error if anything has drifted.

## Setup

On Colab the first cell mounts your Google Drive and works inside the project
folder, so the data and figures persist after the runtime resets; run locally, it
uses the current folder. The next cell then writes the shared module
`berman_core.py` (one source of truth, copied into no notebook) and imports it.

In [15]:
import os
PROJECT = '/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Run_2026-06'
try:
    from google.colab import drive            # on Colab: mount Drive and work in the project folder
    drive.mount('/content/drive')
    os.makedirs(PROJECT, exist_ok=True); os.chdir(PROJECT)
    print('Colab: mounted Drive, working in', os.getcwd())
except ImportError:
    print('Local: working in', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab: mounted Drive, working in /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Run_2026-06


In [16]:
%%writefile berman_core.py
"""
berman_core.py — shared, corrected core for the Berman Modeling rebuild (2026-06).

ONE source of truth for the SAW model, principle sets, Hart baselines, a *seeded*
bootstrap, the activation generators, the weight-tolerance (critical-sigma)
machinery, and the redesigned evaluation criteria. Every notebook %%writefile's
this file and `from berman_core import *`, so a fix here propagates everywhere
(ends the verbatim-copy drift of the old notebooks).

Design decisions baked in (vs. the old code):
  * Determinacy convention is strict |F| > tau, matching the manuscript.
  * RNG is always explicit (np.random.default_rng(seed)); bootstrap is seeded
    and vectorized, so every CI is reproducible.
  * One determinacy cutoff (DEFAULT_TAU) everywhere — no 0.05-here-0.2-there.
  * Corrected category map: federalism is STRUCTURAL, not moral.
  * Criteria are split into REQUIREMENTS (R1-R4, pass/fail, derived from what
    Berman actually claims) and DESCRIPTIVE measures (D1-D2, reported, never used
    to fail the theory). See CONCEPTUAL_AUDIT_vs_Berman.md.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Callable, Tuple

import numpy as np

__all__ = [
    "DEFAULT_TAU", "Principle", "NAMED_PRINCIPLES", "CATEGORY_INDEX",
    "make_weights", "named_weight_vector", "category_of",
    "net_force", "is_determinate", "determinacy_rate",
    "random_activations", "category_correlated_activations", "fully_correlated_activations",
    "bootstrap_ci", "critical_sigma", "CRITICAL_SIGMA_CEIL",
    "hart_hierarchical_determinate", "hart_consensus_determinate", "hart_advantage",
    "hart_validation_status",
    "nonlexicality_score", "ordinalize", "saw_verdict", "lexicographic_verdict",
    "weight_cv", "n_survivors", "cluster_gap",
    "METHODOLOGIES", "methodology_tendencies", "LegalSystem",
    "gmm_bimodal", "bimodality_coefficient",
    "RelianceSystem", "weight_entropy", "gini", "n_bins_occupied",
]

# ---------------------------------------------------------------------------
# Determinacy convention
# ---------------------------------------------------------------------------
DEFAULT_TAU: float = 0.20  # a case is determinate when |F| > tau (strict)


# ---------------------------------------------------------------------------
# Principles and categories  (federalism corrected to 'structural')
# ---------------------------------------------------------------------------
@dataclass
class Principle:
    name: str
    weight: float
    category: str = "structural"  # one of: 'textual', 'structural', 'moral'

    def __post_init__(self) -> None:
        assert 0.0 <= self.weight <= 1.0, f"weight out of [0,1]: {self.weight}"
        assert self.category in ("textual", "structural", "moral"), self.category


# The baseline 9-principle set. Order is fixed; categories are the corrected map.
NAMED_PRINCIPLES: List[Tuple[str, str]] = [
    ("text",       "textual"),
    ("original",   "textual"),
    ("precedent",  "structural"),
    ("structure",  "structural"),
    ("federalism", "structural"),   # <-- was grouped with 'moral' in the old code (bug)
    ("dignity",    "moral"),
    ("equality",   "moral"),
    ("liberty",    "moral"),
    ("welfare",    "moral"),
]

CATEGORY_INDEX: Dict[str, List[int]] = {
    cat: [i for i, (_, c) in enumerate(NAMED_PRINCIPLES) if c == cat]
    for cat in ("textual", "structural", "moral")
}


def category_of(i: int) -> str:
    return NAMED_PRINCIPLES[i][1]


# ---------------------------------------------------------------------------
# Weight generators
# ---------------------------------------------------------------------------
def make_weights(n: int, distribution: str, seed: int) -> np.ndarray:
    """Generate n weights. 'uniform' ~U(.60,.70); 'moderate' ~U(.50,.80);
    'high' ~U(.30,.95). Berman's weights are 'rough' (slight..nearly conclusive)."""
    rng = np.random.default_rng(seed)
    if distribution == "uniform":
        return rng.uniform(0.60, 0.70, n)
    if distribution == "moderate":
        return rng.uniform(0.50, 0.80, n)
    if distribution == "high":
        return rng.uniform(0.30, 0.95, n)
    raise ValueError(f"unknown distribution: {distribution}")


def named_weight_vector(distribution: str = "moderate", seed: int = 0) -> np.ndarray:
    """Weights for the 9 NAMED_PRINCIPLES under a given distribution."""
    return make_weights(len(NAMED_PRINCIPLES), distribution, seed)


# ---------------------------------------------------------------------------
# SAW model (nonlexical aggregation)
# ---------------------------------------------------------------------------
def net_force(weights: np.ndarray, activations: np.ndarray) -> np.ndarray:
    """F = sum_i w_i * a_i. Vectorized over a (..., n) activation array."""
    weights = np.asarray(weights, float)
    activations = np.asarray(activations, float)
    return np.sum(weights * activations, axis=-1)


def is_determinate(weights, activations, tau: float = DEFAULT_TAU):
    return np.abs(net_force(weights, activations)) > tau


def determinacy_rate(weights, activation_matrix, tau: float = DEFAULT_TAU) -> float:
    return float(np.mean(is_determinate(weights, activation_matrix, tau)))


# ---------------------------------------------------------------------------
# Activation generators (the conservative independent baseline + realistic structure)
# ---------------------------------------------------------------------------
def random_activations(n_cases: int, n_principles: int, rng: np.random.Generator) -> np.ndarray:
    """Independent activations ~ U(-1, 1). The conservative baseline."""
    return rng.uniform(-1.0, 1.0, size=(n_cases, n_principles))


def category_correlated_activations(n_cases: int, rng: np.random.Generator,
                                    rho: float = 0.7) -> np.ndarray:
    """Activations for the 9 NAMED_PRINCIPLES where principles in the same
    category share a common latent factor (textual together, structural together,
    moral together)."""
    n_p = len(NAMED_PRINCIPLES)
    a = np.empty((n_cases, n_p))
    for cat, idx in CATEGORY_INDEX.items():
        factor = rng.uniform(-1.0, 1.0, size=(n_cases, 1))
        idio = rng.uniform(-1.0, 1.0, size=(n_cases, len(idx)))
        a[:, idx] = np.clip(rho * factor + (1 - rho) * idio, -1.0, 1.0)
    return a


def fully_correlated_activations(n_cases: int, n_principles: int,
                                 rng: np.random.Generator, rho: float = 0.7) -> np.ndarray:
    """All principles share one latent factor."""
    factor = rng.uniform(-1.0, 1.0, size=(n_cases, 1))
    idio = rng.uniform(-1.0, 1.0, size=(n_cases, n_principles))
    return np.clip(rho * factor + (1 - rho) * idio, -1.0, 1.0)


# ---------------------------------------------------------------------------
# Bootstrap — SEEDED and vectorized (every CI is reproducible)
# ---------------------------------------------------------------------------
def bootstrap_ci(values, n_boot: int = 2000, ci: int = 95, seed: int = 0):
    """Return (mean, lo, hi) for a percentile bootstrap CI on the mean."""
    rng = np.random.default_rng(seed)
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return (float("nan"),) * 3
    means = rng.choice(arr, size=(n_boot, arr.size), replace=True).mean(axis=1)
    lo = float(np.percentile(means, (100 - ci) / 2))
    hi = float(np.percentile(means, 100 - (100 - ci) / 2))
    return float(arr.mean()), lo, hi


# ---------------------------------------------------------------------------
# Weight-uncertainty tolerance: critical sigma  (R2 — "rough weights suffice")
# ---------------------------------------------------------------------------
CRITICAL_SIGMA_CEIL = 1.0  # grid maximum; a returned value == this is right-censored


def critical_sigma(weights: np.ndarray, activation_matrix: np.ndarray,
                   tau: float = DEFAULT_TAU, accuracy: float = 0.80,
                   robust_thresh: float = 0.90, n_mc: int = 64, seed: int = 0,
                   sigma_grid: Optional[np.ndarray] = None) -> float:
    """Smallest weight-noise sigma at which fewer than `accuracy` of cases remain
    individually robust.

    Two-level definition: add N(0, sigma^2) to each weight (clipped to [0,1]); a
    *case* is robust at sigma if it keeps its unperturbed signed VERDICT
    (determinate-for / determinate-against / indeterminate) on at least
    `robust_thresh` of MC draws; sigma* is the smallest sigma at which the
    fraction of robust cases falls below `accuracy`.

    Verdict-stability (not just determinate/indeterminate class) so that a noise
    level that flips a determinate-for case to determinate-against counts as
    instability. The grid runs to CRITICAL_SIGMA_CEIL; a returned value equal to
    the ceiling is right-censored (true tolerance is >= ceiling) -- callers
    should flag these rather than average them as if measured.
    """
    rng = np.random.default_rng(seed)
    w = np.asarray(weights, float)
    A = np.asarray(activation_matrix, float)
    n_p = w.size
    if sigma_grid is None:
        sigma_grid = np.linspace(0.0, CRITICAL_SIGMA_CEIL, 51)

    true_F = A @ w
    true_v = np.sign(true_F); true_v[np.abs(true_F) <= tau] = 0    # (cases,) verdict
    noise = rng.standard_normal((n_mc, A.shape[0], n_p))          # (mc, cases, p)

    prev_sigma, prev_surv = 0.0, 1.0
    for sigma in sigma_grid:
        if sigma == 0:
            prev_sigma, prev_surv = 0.0, 1.0
            continue
        w_noisy = np.clip(w[None, None, :] + sigma * noise, 0.0, 1.0)
        F = np.einsum("mcp,cp->mc", w_noisy, A)                   # (mc, cases)
        v = np.sign(F); v[np.abs(F) <= tau] = 0
        agree_frac = np.mean(v == true_v[None, :], axis=0)        # per case
        survival = float(np.mean(agree_frac >= robust_thresh))
        if survival < accuracy:
            frac = (accuracy - prev_surv) / (survival - prev_surv)
            return float(prev_sigma + frac * (sigma - prev_sigma))
        prev_sigma, prev_surv = sigma, survival
    return float(sigma_grid[-1])  # right-censored at the ceiling


# ---------------------------------------------------------------------------
# Hart baselines  (R4 — "more law than Hart", reported as a margin/curve)
# ---------------------------------------------------------------------------
def hart_consensus_determinate(activations, frac: float = 0.6,
                               act_cut: float = 0.1) -> np.ndarray:
    """Consensus Hart (the faithful operationalization of Hart-as-validation):
    a case is determinate only when the share of *active* principles pointing the
    same direction exceeds `frac`. Hart requires agreement; below `frac` the law
    'runs out'."""
    A = np.atleast_2d(np.asarray(activations, float))
    pos = np.sum(A > act_cut, axis=1)
    neg = np.sum(A < -act_cut, axis=1)
    active = pos + neg
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.maximum(pos, neg) / np.where(active == 0, 1, active)
    return (active > 0) & (share > frac)


def hart_hierarchical_determinate(weights, activations, tier_of: np.ndarray,
                                  frac: float = 0.6, act_cut: float = 0.1,
                                  tau: float = DEFAULT_TAU) -> np.ndarray:
    """Charitable *lexical-priority* Hart: principles sit in priority tiers
    (0 = top). For each case, the highest-priority tier that has ANY active
    principle is DISPOSITIVE: that tier decides iff its active principles reach
    consensus (>= `frac` agree in sign); otherwise the law runs out. Lower tiers
    are NOT consulted once a higher tier is reached — that is the lexical priority
    Berman's nonlexical model is contrasted against, and it is what stops Hart
    from finding law in nearly every case via tier escape-hatches.
    """
    A = np.atleast_2d(np.asarray(activations, float))
    tier_of = np.asarray(tier_of, int)
    out = np.zeros(A.shape[0], dtype=bool)
    reached = np.zeros(A.shape[0], dtype=bool)  # a dispositive tier has been reached
    for t in np.unique(tier_of):
        idx = np.where(tier_of == t)[0]
        sub = A[:, idx]
        pos = np.sum(sub > act_cut, axis=1)
        neg = np.sum(sub < -act_cut, axis=1)
        active = pos + neg
        this_tier = (active > 0) & (~reached)        # cases first resolved at this tier
        with np.errstate(invalid="ignore", divide="ignore"):
            share = np.maximum(pos, neg) / np.where(active == 0, 1, active)
        out |= this_tier & (share > frac)             # decided iff consensus here (strict, matches consensus Hart)
        reached |= this_tier                           # dispositive; no escape downward
    return out


def hart_validation_status(activations, k: int = 2, act_cut: float = 0.5) -> np.ndarray:
    """Naive validation foil used to make Berman's 'no contradiction' point.

    A direction counts as independently 'validated' when at least `k` principles
    are strongly activated (|a| > act_cut) in that direction. Returns one integer
    code per case:
        0 = gap          (neither direction validated -> the law runs out)
        1 = for          (only the positive direction validated)
        2 = against      (only the negative direction validated)
        3 = contradiction(both directions validated -> q and not-q are both law)

    The point: code 3 is possible for a validation model, but impossible for
    Berman's aggregation, because a single net force has a single sign. So the sum
    never returns code 3.
    """
    A = np.atleast_2d(np.asarray(activations, float))
    pos = np.sum(A > act_cut, axis=1) >= k
    neg = np.sum(A < -act_cut, axis=1) >= k
    code = np.zeros(A.shape[0], dtype=int)
    code[pos & ~neg] = 1
    code[neg & ~pos] = 2
    code[pos & neg] = 3
    return code


def hart_advantage(weights, activation_matrix, tier_of, frac: float = 0.6,
                   tau: float = DEFAULT_TAU) -> float:
    """Berman determinacy rate minus hierarchical-Hart determinacy rate (a margin,
    not a pass/fail)."""
    berman = determinacy_rate(weights, activation_matrix, tau)
    hart = float(np.mean(hart_hierarchical_determinate(
        weights, activation_matrix, tier_of, frac=frac, tau=tau)))
    return berman - hart


# ---------------------------------------------------------------------------
# R3 — nonlexicality.  Does determination track activation MAGNITUDES (and many
# principles jointly), as Berman requires, rather than just signs/ranks (lexical)?
# ---------------------------------------------------------------------------
def ordinalize(weights: np.ndarray) -> np.ndarray:
    """Replace weights by rank-based pseudo-weights (1..n)/n — discards cardinal info."""
    order = np.argsort(np.argsort(weights))  # ranks 0..n-1
    return (order + 1) / len(weights)


# Verdict functions return an integer verdict per case: +1 / -1 (determinate for
# / against) or 0 (indeterminate). These are the inputs to the R3 nonlexicality
# probe and are reused by the aggregation-function suite.
def saw_verdict(weights, activation_matrix, tau: float = DEFAULT_TAU) -> np.ndarray:
    """Nonlexical: verdict from the *magnitude-weighted sum* of all principles."""
    F = np.asarray(activation_matrix, float) @ np.asarray(weights, float)
    v = np.sign(F)
    v[np.abs(F) <= tau] = 0
    return v.astype(int)


def lexicographic_verdict(weights, activation_matrix, act_cut: float = 0.05) -> np.ndarray:
    """Lexical: verdict from the SIGN of the single highest-weight active
    principle (Raz exclusionary-reasons style). This is the mode Berman's theory
    is defined against — included only as the lexical foil for the R3 probe."""
    w = np.asarray(weights, float)
    A = np.atleast_2d(np.asarray(activation_matrix, float))
    order = np.argsort(-w)  # highest weight first
    out = np.zeros(A.shape[0], dtype=int)
    decided = np.zeros(A.shape[0], dtype=bool)
    for j in order:
        active = (np.abs(A[:, j]) > act_cut) & (~decided)
        out[active] = np.sign(A[active, j]).astype(int)
        decided |= active
    return out


def nonlexicality_score(verdict_fn: Callable[[np.ndarray], np.ndarray],
                        activation_matrix: np.ndarray,
                        seed: int = 0, n_mc: int = 32, jitter: float = 0.20) -> float:
    """R3 probe. Fraction of cases whose VERDICT changes under a magnitude-only
    perturbation of activations that preserves every activation's sign AND the
    within-case rank order of |activation|.

    A lexical rule (decides on the top-ranked active principle) is invariant to
    such perturbations -> score ~ 0. A genuinely nonlexical, magnitude-aggregating
    rule (SAW) is sensitive -> score > 0. `verdict_fn(A)` maps an (n,9) activation
    matrix to an integer verdict array in {-1,0,+1}.
    """
    rng = np.random.default_rng(seed)
    A = np.asarray(activation_matrix, float)
    base = verdict_fn(A)
    rank = np.argsort(np.argsort(np.abs(A), axis=1), axis=1)  # rank of |a| within row
    n_p = A.shape[1]
    changed = np.zeros(A.shape[0])
    for _ in range(n_mc):
        mag = np.clip(np.abs(A) * (1 + rng.uniform(-jitter, jitter, A.shape)), 0, 1)
        sorted_mag = np.sort(mag, axis=1)
        # Strictly-increasing epsilon breaks ties created when two magnitudes both
        # clip to 1.0; without this, re-placing tied values by index can silently
        # change the rank order (and thus the decision of a lexical rule). This
        # guarantees |A2| keeps the ORIGINAL within-row rank order exactly.
        sorted_mag = sorted_mag + 1e-9 * np.arange(n_p)
        A2 = np.sign(A) * np.take_along_axis(sorted_mag, rank, axis=1)
        changed += (verdict_fn(A2) != base)
    return float(np.mean(changed / n_mc))


# ---------------------------------------------------------------------------
# Descriptive weight statistics (D1 gradation, and emergence diagnostics)
# ---------------------------------------------------------------------------
def weight_cv(weights: np.ndarray) -> float:
    w = np.asarray(weights, float)
    m = w.mean()
    return float(w.std() / m) if m > 0 else 0.0


def n_survivors(weights: np.ndarray, cutoff: float = 0.05) -> int:
    return int(np.sum(np.asarray(weights, float) > cutoff))


def cluster_gap(weights: np.ndarray) -> float:
    """Gap between the two k-means clusters of weights (bimodality diagnostic)."""
    w = np.sort(np.asarray(weights, float))
    if w.size < 2:
        return 0.0
    best = None
    for k in range(1, w.size):
        lo, hi = w[:k], w[k:]
        gap = hi.mean() - lo.mean()
        if best is None or gap > best:
            best = gap
    return float(best)


# ===========================================================================
# Agent-based model of weight emergence (Notebook 1)
# ===========================================================================
# A bench of judges, each carrying an interpretive methodology that sets how
# often it invokes each principle. A principle's emergent "uptake share" is how
# often the bench takes it up across decisions. This is a deliberately simple
# model of Berman's "taking-up" (JESP p. 302); it is NOT his full account of
# weight, which is conferred by reliance and by winning conflicts (pp. 322, 324)
# and is taken up in Notebook 1c. The emergent quantity here is called "uptake
# share," never "weight in Berman's sense."

METHODOLOGIES = ("textualist", "purposivist", "pragmatist", "originalist")

# For each methodology, the principles it tends to invoke often ('high') or
# rarely ('low'); anything unlisted is 'medium'. Names refer to NAMED_PRINCIPLES.
_METH_SETS = {
    "textualist":  {"high": {"text", "original"},
                    "low":  {"dignity", "equality", "liberty", "welfare"}},
    "purposivist": {"high": {"dignity", "equality", "liberty"},
                    "low":  {"text", "original", "federalism"}},
    "pragmatist":  {"high": {"welfare", "precedent"},
                    "low":  {"dignity", "equality", "original"}},
    "originalist": {"high": {"original", "text", "federalism"},
                    "low":  {"dignity", "equality", "liberty", "welfare"}},
}

# Invocation-probability ranges (high, medium, low) by how specialized the
# methodologies are. 'wide' = strong specialization (headline); 'narrow' = nearly
# homogeneous (used to show heterogeneity is necessary for the split).
_SPREAD_RANGES = {
    "wide":   ((0.85, 0.95), (0.35, 0.65), (0.02, 0.08)),
    "medium": ((0.70, 0.80), (0.40, 0.60), (0.15, 0.25)),
    "narrow": ((0.60, 0.70), (0.42, 0.58), (0.30, 0.40)),
}


def methodology_tendencies(methodology: str, spread: str, rng) -> np.ndarray:
    """A 9-vector of per-principle invocation probabilities for one judge of a
    given methodology, drawn from the high/medium/low ranges set by `spread`."""
    hi, med, lo = _SPREAD_RANGES[spread]
    sets = _METH_SETS[methodology]
    t = np.empty(len(NAMED_PRINCIPLES))
    for i, (name, _) in enumerate(NAMED_PRINCIPLES):
        rng_range = hi if name in sets["high"] else (lo if name in sets["low"] else med)
        t[i] = rng.uniform(*rng_range)
    return t


class LegalSystem:
    """A bench of judges whose case-by-case invocations produce emergent
    principle uptake-shares.

    `turnover` selects how judges are replaced over time:
      'static'      no replacement (the headline emergence must hold here);
      'fitness'     replace the least decisive 20% with mutated copies of the
                    most decisive 20% (the old, suspect, non-Bermanian rule);
      'appointment' replace a random 20%, reseated from the methodology pool;
      'noise'       replace a random 20% with mutated copies of random judges.
    The point of the four regimes is to show the emergent structure does NOT
    depend on the turnover rule.
    """

    def __init__(self, n_agents=100, methodology_mix=None, spread="wide",
                 use_threshold=True, threshold_pct=50, turnover="static",
                 boost_cut=0.3, learn_from_system=0.15, seed=0):
        # One master seed -> five independent streams. No global np.random.
        ss = np.random.SeedSequence(seed)
        (self._rng_agents, self._rng_panels, self._rng_cases,
         self._rng_dec, self._rng_rep) = (np.random.default_rng(s) for s in ss.spawn(5))
        self.n_agents = n_agents
        self.spread = spread
        self.use_threshold = use_threshold
        self.threshold_pct = threshold_pct
        self.turnover = turnover
        self.boost_cut = boost_cut
        self.learn_from_system = learn_from_system
        self.n_p = len(NAMED_PRINCIPLES)

        # Assign methodologies exactly (round-robin or a given mix), then permute
        # the arrangement with the seeded generator (never np.random.shuffle).
        if methodology_mix is None:
            base = [METHODOLOGIES[i % 4] for i in range(n_agents)]
        else:
            base = []
            for m, frac in methodology_mix.items():
                base += [m] * int(round(frac * n_agents))
            base = (base + [METHODOLOGIES[0]] * n_agents)[:n_agents]
        self.methodology = list(self._rng_agents.permutation(base))
        self.tendencies = np.array([methodology_tendencies(m, spread, self._rng_agents)
                                    for m in self.methodology])     # (n_agents, 9)
        self.system_weights = np.full(self.n_p, 0.5)                # uniform init
        self.raw_uptake = np.full(self.n_p, 0.5)                    # pre-threshold share
        self.success = np.zeros(n_agents)
        self.decisions = np.zeros(n_agents)
        self.trajectory = []

    def _recompute_weights(self, counts, total):
        raw = counts / max(total, 1)
        raw = raw / max(raw.max(), 1e-12)                           # normalize to max
        self.raw_uptake = raw.copy()                               # the pre-threshold vector
        w = raw.copy()
        if self.use_threshold:                                     # zero the bottom percentile
            cut = np.percentile(w, self.threshold_pct)
            w = np.where(w < cut, 0.0, w)
            if w.max() > 0:
                w = w / w.max()
        self.system_weights = w

    def _replace(self, idx_out, idx_template):
        for o, t in zip(idx_out, idx_template):
            self.methodology[o] = self.methodology[t]
            self.tendencies[o] = np.clip(self.tendencies[t] + self._rng_rep.normal(0, 0.03, self.n_p),
                                         0.02, 0.98)
            self.success[o] = 0.0
            self.decisions[o] = 0.0

    def _turnover_step(self):
        k = max(1, int(0.2 * self.n_agents))
        if self.turnover == "static":
            return
        if self.turnover == "fitness":
            # eligible only with >= 5 decisions, so 1-2-case luck cannot rank an agent
            rate = np.where(self.decisions >= 5, self.success / np.maximum(self.decisions, 1), -1.0)
            order = np.argsort(rate)
            self._replace(order[:k], order[-k:])                   # bottom k <- top k
        elif self.turnover == "appointment":
            for o in self._rng_rep.choice(self.n_agents, k, replace=False):
                m = METHODOLOGIES[int(self._rng_rep.integers(4))]
                self.methodology[o] = m
                self.tendencies[o] = methodology_tendencies(m, self.spread, self._rng_rep)
                self.success[o] = 0.0
                self.decisions[o] = 0.0
        elif self.turnover == "noise":
            self._replace(self._rng_rep.choice(self.n_agents, k, replace=False),
                          self._rng_rep.choice(self.n_agents, k, replace=False))

    def simulate(self, n_cases=2000, panel_size=9, update_every=100, selection_every=500):
        counts = np.zeros(self.n_p)
        total = 0
        A = self._rng_cases.uniform(-1, 1, size=(n_cases, self.n_p))           # case stream up front
        panels = np.array([self._rng_panels.choice(self.n_agents, panel_size, replace=False)
                           for _ in range(n_cases)])
        self.trajectory = []
        for c in range(n_cases):
            a = A[c]
            panel = panels[c]
            tend = self.tendencies[panel]                                      # (panel, 9)
            prob = 0.85 * tend + self.learn_from_system * self.system_weights[None, :]
            boost = np.abs(a) > self.boost_cut                                 # strong activations invoked more
            prob = np.where(boost[None, :], np.minimum(prob * 1.15, 0.99), prob)
            invoked = self._rng_dec.uniform(size=(panel_size, self.n_p)) < prob
            counts += invoked.sum(axis=0)
            total += int(invoked.sum())
            forces = (invoked * self.system_weights[None, :] * a[None, :]).sum(axis=1)  # SAW per judge
            med = np.median(np.abs(forces))
            self.success[panel] += (np.abs(forces) >= med)                     # fitness bookkeeping
            self.decisions[panel] += 1
            if (c + 1) % update_every == 0:
                self._recompute_weights(counts, total)
                # record the smooth pre-threshold uptake for convergence/trajectory
                self.trajectory.append((c + 1, self.raw_uptake.copy()))
            if (c + 1) % selection_every == 0:
                self._turnover_step()
        self._recompute_weights(counts, total)
        return self.system_weights


def gmm_bimodal(values, seed=0, dbic_thresh=10.0):
    """Two-vs-one Gaussian-mixture BIC test for bimodality. Returns
    (dbic, separated, is_bimodal): dbic = BIC(1) - BIC(2) (larger favors two
    components); `separated` requires the two means be at least one larger-
    component SD apart (rejects two overlapping Gaussians). Deterministic."""
    from sklearn.mixture import GaussianMixture
    x = np.asarray(values, float).reshape(-1, 1)
    if len(np.unique(np.round(x, 6))) < 2:
        return 0.0, False, False
    g1 = GaussianMixture(1, random_state=seed, n_init=5).fit(x)
    g2 = GaussianMixture(2, random_state=seed, n_init=5).fit(x)
    dbic = float(g1.bic(x) - g2.bic(x))
    means = g2.means_.ravel()
    sds = np.sqrt(g2.covariances_.ravel())
    separated = bool(abs(means[0] - means[1]) > sds.max())
    return dbic, separated, bool(dbic > dbic_thresh and separated)


def bimodality_coefficient(values):
    """Sarle's bimodality coefficient. > 0.555 suggests bimodality. Underpowered
    for very small n, so use it on pooled data, not a single 9-vector."""
    from scipy import stats as _st
    x = np.asarray(values, float)
    n = x.size
    if n < 4:
        return float("nan")
    g = _st.skew(x); k = _st.kurtosis(x, fisher=True)
    return float((g**2 + 1) / (k + 3 * (n - 1)**2 / ((n - 2) * (n - 3))))


# ===========================================================================
# Reliance / by-battle weight conferral (Notebook 2)
# ===========================================================================
# Berman's conferred weight (JESP pp. 322, 324): a principle gains weight by
# winning CONTESTED, DISPOSITIVE battles, amplified by how much the deciding
# precedent is later RELIED UPON (cited), decayed gradually, and removed when a
# precedent is OVERRULED. Battles are decided by a FIXED exogenous "pressure"
# structure (the practice), NOT by the conferred weight being built: making it
# literally circular ("weight made so by battle") produces runaway lock-in, so
# the decision and the read-out are decoupled. There is NO weight floor -- that
# is what made the old "precedent stock" collapse to uniform weights.

def weight_entropy(w):
    """Shannon entropy of a normalized weight vector (higher = more spread)."""
    w = np.asarray(w, float); s = w.sum()
    if s <= 0:
        return 0.0
    p = w / s; p = p[p > 0]
    return float(-(p * np.log(p)).sum())


def gini(w):
    """Gini coefficient of a weight vector (0 = equal, ->1 = concentrated)."""
    w = np.sort(np.asarray(w, float)); n = len(w)
    if w.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1).dot(w) / (n * w.sum()))


def n_bins_occupied(w, edges=(0.2, 0.4, 0.6, 0.8)):
    """How many of Berman's ~5 rough weight bands a weight vector occupies."""
    return int(len(np.unique(np.digitize(np.asarray(w, float), edges))))


class RelianceSystem:
    """Conferred weight as reliance-amplified, decayed, overruling-prunable
    battle-victories. Battles are decided by `pressure_weights` (fixed), and the
    conferred weight is read out separately, so the dynamics cannot lock in."""

    def __init__(self, pressure_weights=None, invoke_prob=None, salience=None,
                 activations="correlated", cite_mode="similarity", n_cite=3,
                 overrule_rate=0.02, decay_rate=0.9995, clash_cut=0.10,
                 feedback_alpha=0.0, tau=DEFAULT_TAU, seed=0):
        # feedback_alpha > 0 feeds the running conferred weight back into the
        # battle decision (the literally-circular "made so by battle"). It exists
        # only to DEMONSTRATE the runaway lock-in that justifies the decoupled
        # design; the headline runs use feedback_alpha = 0.
        ss = np.random.SeedSequence(seed)
        (self._rng_cases, self._rng_cite, self._rng_over,
         self._rng_inv, self._rng_setup) = (np.random.default_rng(s) for s in ss.spawn(5))
        n_p = len(NAMED_PRINCIPLES)
        self.n_p = n_p
        self.pressure_weights = (named_weight_vector("high", seed) if pressure_weights is None
                                 else np.asarray(pressure_weights, float))
        # uptake invocation propensity. It tracks the same exogenous PRESSURE
        # structure that decides battles (a principle pressed harder is invoked
        # more and can win more), so uptake and conferred weight are correlated
        # but distinct. It is NOT a function of the conferred weight being built,
        # so the two can still come apart (the forced-loser demo overrides it).
        self.invoke_prob = (np.clip(0.2 + 0.7 * (self.pressure_weights / self.pressure_weights.max())
                                    + self._rng_setup.normal(0, 0.05, n_p), 0.05, 0.98)
                            if invoke_prob is None else np.asarray(invoke_prob, float))
        self.salience = np.ones(n_p) if salience is None else np.asarray(salience, float)
        self.activations, self.cite_mode, self.n_cite = activations, cite_mode, n_cite
        self.overrule_rate, self.decay_rate, self.clash_cut, self.tau = \
            overrule_rate, decay_rate, clash_cut, tau
        self.feedback_alpha = feedback_alpha
        self.invoked_total = np.zeros(n_p)
        self.n_seen = 0
        self._ns = 0

    def run(self, n_cases=2000):
        np_ = self.n_p
        A = (category_correlated_activations(n_cases, self._rng_cases) if self.activations == "correlated"
             else random_activations(n_cases, np_, self._rng_cases))
        A = np.clip(A * self.salience[None, :], -1, 1)
        inv = self._rng_inv.uniform(size=(n_cases, np_))
        act = np.zeros((n_cases, np_)); winners = np.zeros((n_cases, np_), bool)
        verdict = np.zeros(n_cases); margin = np.zeros(n_cases); opposing = np.zeros(n_cases)
        contested = np.zeros(n_cases, bool); cc = np.zeros(n_cases)
        good = np.zeros(n_cases, bool); born = np.zeros(n_cases, int); nrm = np.zeros(n_cases)
        ns = 0
        eff = self.pressure_weights.copy()                        # effective decision pressure
        for c in range(n_cases):
            a = A[c]
            self.invoked_total += (inv[c] < self.invoke_prob)     # uptake (weight-independent)
            self.n_seen += 1
            contrib = eff * a
            F = contrib.sum()
            if abs(F) <= self.tau:
                continue                                          # equipoise: no victor
            v = 1.0 if F > 0 else -1.0
            big = np.abs(contrib) > self.clash_cut
            win = (np.sign(contrib) == v) & big                  # dispositive winners only
            cont = bool((contrib > self.clash_cut).any() and (contrib < -self.clash_cut).any())
            opp = float(np.abs(contrib[(np.sign(contrib) != v) & big]).sum())
            act[ns] = a; winners[ns] = win; verdict[ns] = v; margin[ns] = abs(F)
            opposing[ns] = opp; contested[ns] = cont; born[ns] = c
            nrm[ns] = np.linalg.norm(a); good[ns] = True
            if ns > 0:                                           # reliance: cite prior good precedents
                gi = np.where(good[:ns])[0]
                if len(gi):
                    if self.cite_mode == "preferential":
                        order = gi[np.argsort(-cc[gi])]
                    elif self.cite_mode == "random":
                        order = self._rng_cite.permutation(gi)
                    else:                                        # similarity / blended (cosine of facts)
                        sims = (act[gi] @ a) / (nrm[gi] * np.linalg.norm(a) + 1e-9)
                        score = sims * (1 + cc[gi]) if self.cite_mode == "blended" else sims
                        order = gi[np.argsort(-score)]
                    cc[order[:self.n_cite]] += 1
                    if self._rng_over.uniform() < self.overrule_rate:   # overrule a contradictory good precedent
                        sims = (act[gi] @ a) / (nrm[gi] * np.linalg.norm(a) + 1e-9)
                        cand = gi[(verdict[gi] == -v) & (sims > 0.5)]
                        if len(cand):
                            good[cand[0]] = False
            ns += 1
            if self.feedback_alpha > 0 and (c + 1) % 100 == 0 and ns > 0:
                mm = good[:ns] & contested[:ns]                   # lock-in demo: feed weight back into decisions
                if mm.any():
                    dec = self.decay_rate ** ((c + 1) - born[:ns][mm])
                    eq = opposing[:ns][mm] / (margin[:ns][mm] + opposing[:ns][mm] + 1e-9)
                    amp = margin[:ns][mm] * (1 + cc[:ns][mm]) * eq * dec
                    cur = (amp[:, None] * winners[:ns][mm]).sum(axis=0)
                    if cur.max() > 0:
                        eff = (1 - self.feedback_alpha) * self.pressure_weights + \
                              self.feedback_alpha * (cur / cur.max())
        self._act, self._winners, self._verdict = act[:ns], winners[:ns], verdict[:ns]
        self._margin, self._opposing, self._contested = margin[:ns], opposing[:ns], contested[:ns]
        self._cc, self._good, self._born = cc[:ns], good[:ns], born[:ns]
        self._n_cases, self._ns = n_cases, ns
        return self.read_weights()

    def _readout(self, use_reliance=True, use_equipoise=True):
        if self._ns == 0:
            return np.zeros(self.n_p)
        m = self._good & self._contested
        if not m.any():
            return np.zeros(self.n_p)
        decay = self.decay_rate ** (self._n_cases - self._born[m])
        equ = (self._opposing[m] / (self._margin[m] + self._opposing[m] + 1e-9)) if use_equipoise else 1.0
        rel = (1 + self._cc[m]) if use_reliance else 1.0
        amp = self._margin[m] * rel * equ * decay
        conf = (amp[:, None] * self._winners[m]).sum(axis=0)
        mx = conf.max()
        return conf / mx if mx > 0 else conf

    def read_weights(self):
        return self._readout(True, True)

    def read_weights_no_reliance(self):
        return self._readout(False, True)

    @property
    def uptake_share(self):
        return self.invoked_total / max(self.n_seen, 1)


if __name__ == "__main__":
    # quick self-test: static properties
    rng = np.random.default_rng(0)
    w = named_weight_vector("moderate", seed=0)
    A = random_activations(2000, 9, rng)
    print("categories:", CATEGORY_INDEX)
    print("determinacy rate (tau=0.2):", round(determinacy_rate(w, A), 4))
    print("critical sigma:", round(critical_sigma(w, A), 4))
    print("nonlexicality SAW vs lexical:",
          round(nonlexicality_score(lambda X: saw_verdict(w, X), A), 4),
          round(nonlexicality_score(lambda X: lexicographic_verdict(w, X), A), 4))
    # quick self-test: ABM emergence
    sysm = LegalSystem(turnover="static", seed=0)
    we = sysm.simulate(n_cases=2000)
    print("\nABM emergent uptake (post-threshold):", np.round(we, 2))
    print("ABM raw uptake (pre-threshold):", np.round(sysm.raw_uptake, 2))
    print("emerged (!= uniform):", not np.allclose(sysm.raw_uptake, 0.5))
    print("raw-uptake bimodal (GMM):", gmm_bimodal(sysm.raw_uptake))
    print("cluster gap:", round(cluster_gap(sysm.raw_uptake), 3), "| CV:", round(weight_cv(sysm.raw_uptake), 3))


Overwriting berman_core.py


In [17]:
import os, json, itertools
import numpy as np, pandas as pd
from berman_core import *

# Where results are written. Override with the BERMAN_OUTDIR environment variable
# (e.g. to a mounted Drive folder on Colab); otherwise a local 'data' folder.
OUTDIR = os.environ.get('BERMAN_OUTDIR', 'data/notebook0')
os.makedirs(OUTDIR, exist_ok=True)

SEEDS = 30                 # independent repetitions per configuration
summary = {}               # headline numbers, written to JSON at the end
print('Core loaded. Principle categories:', CATEGORY_INDEX)

Core loaded. Principle categories: {'textual': [0, 1], 'structural': [2, 3, 4], 'moral': [5, 6, 7, 8]}


## R2 — Rough weights suffice

Perry's objection to any aggregation model is that practice cannot pin down the
weights precisely enough, so the sums will be too noisy to yield determinate law.
Berman's reply is that the weights only need to be roughly right.

To test this we take a set of weights, add Gaussian noise of standard deviation
$\sigma$ to every weight, and check whether each case keeps its verdict. The
**critical uncertainty** $\sigma^{*}$ is the noise level at which one case in
five starts to flip. A larger $\sigma^{*}$ means the model tolerates more
imprecision. (`critical_sigma` compares the full signed verdict, so a flip from
determinate-for to determinate-against counts as a change, not just a switch
between determinate and indeterminate.)

The tolerance depends on the weight spread, the number of principles, and the
threshold, so a single number would mislead. We sweep all 45 combinations and
then ask which of the three choices moves $\sigma^{*}$ the most.

In [18]:
# 45 configurations: 3 weight spreads x 3 principle counts x 5 thresholds.
dists  = ['uniform', 'moderate', 'high']    # how spread out the weights are
counts = [5, 9, 15]                         # number of principles
taus   = [0.05, 0.10, 0.15, 0.20, 0.30]     # determinacy thresholds

rows = []
for dist, n, tau in itertools.product(dists, counts, taus):
    sig = []
    for s in range(SEEDS):
        w = make_weights(n, dist, seed=s)                         # stipulated weights
        A = random_activations(300, n, np.random.default_rng(1000 + s))  # 300 synthetic cases
        sig.append(critical_sigma(w, A, tau=tau, n_mc=40, seed=s))       # tolerance for this config
    m, lo, hi = bootstrap_ci(sig, seed=0)                          # mean + 95% CI across seeds
    rows.append(dict(distribution=dist, n=n, tau=tau, sigma=m, lo=lo, hi=hi))
df1 = pd.DataFrame(rows); df1.to_csv(f'{OUTDIR}/exp1_tolerance.csv', index=False)

# How far the mean sigma* moves when we vary each design choice on its own
# (max minus min of the group means). The larger the spread, the more that
# choice matters.
spread = lambda s: float(s.max() - s.min())
sp_tau = df1.groupby('tau').sigma.mean()
sp_n   = df1.groupby('n').sigma.mean()
sp_d   = df1.groupby('distribution').sigma.mean()

# A returned sigma* equal to the grid ceiling is right-censored (the true value
# is larger but unmeasured). Count these so they are not read as exact.
n_cens = int((df1.sigma >= CRITICAL_SIGMA_CEIL - 1e-9).sum())

summary['exp1'] = dict(mean_sigma=float(df1.sigma.mean()),
                       sigma_min=float(df1.sigma.min()), sigma_max=float(df1.sigma.max()),
                       n_censored=n_cens, n_configs=len(df1),
                       spread_tau=spread(sp_tau), spread_count=spread(sp_n), spread_dist=spread(sp_d))
print(f'mean critical sigma = {df1.sigma.mean():.3f}  (range {df1.sigma.min():.3f}-{df1.sigma.max():.3f})')
print(f'{n_cens}/{len(df1)} configurations right-censored at the ceiling {CRITICAL_SIGMA_CEIL}')
print(f'spread from threshold = {spread(sp_tau):.3f} | from count = {spread(sp_n):.3f} | from spread = {spread(sp_d):.3f}')
print('threshold accounts for {:.0%} of the combined spread'.format(
    spread(sp_tau) / (spread(sp_tau) + spread(sp_n) + spread(sp_d))))

mean critical sigma = 0.090  (range 0.071-0.134)
0/45 configurations right-censored at the ceiling 1.0
spread from threshold = 0.049 | from count = 0.008 | from spread = 0.001
threshold accounts for 84% of the combined spread


*Reading.* The mean tolerance is substantial, which supports Berman against
Perry. The threshold choice moves $\sigma^{*}$ far more than the weight spread or
the principle count do. Two notes for the manuscript: the threshold is the one
parameter Berman leaves unspecified, yet it dominates; and the principle count
barely matters, which is a mild tension with Berman's "sheer numbers" version of
the reply (he expects many aligned principles to swamp weight error).

## R4 — More determinate law than Hart

Berman's model is meant to answer the "too little law" worry: a Hart-style model
that recognizes law only where officials converge finds no law in hard cases,
where they do not converge. Aggregation can still return a verdict there, because
weak support from several principles can add up.

We compare the model's determinacy rate against two Hart-style foils:

- **Consensus Hart.** The case is determinate only when more than a set fraction
  of the active principles point the same way. We report 50, 60, 70, and 80
  percent. This is Hart's requirement that the criteria converge.
- **Lexical-priority Hart.** Principles sit in priority tiers (textual, then
  structural, then moral). The highest tier with any active principle decides,
  and only if its principles agree; lower tiers are never consulted. This is the
  rigid if-then structure Berman contrasts his model against.

We report the **margin** (Berman minus Hart), not a single pass/fail number,
because the margin depends on how much agreement Hart is made to require, and
that dependence is part of the point.

In [19]:
# Priority tiers for the lexical Hart foil: textual is consulted first, then
# structural, then moral. (Tier index 0 = highest priority.)
tier_map = {'textual': 0, 'structural': 1, 'moral': 2}
tier_of  = np.array([tier_map[c] for _, c in NAMED_PRINCIPLES])

rows = []
for s in range(SEEDS):
    w = named_weight_vector('moderate', seed=s)                    # the 9 named principles
    A = random_activations(1000, 9, np.random.default_rng(2000 + s))
    rec = dict(seed=s,
               berman=determinacy_rate(w, A, 0.2),
               hart_hier=float(np.mean(hart_hierarchical_determinate(w, A, tier_of, frac=0.6, tau=0.2))))
    for f in (0.5, 0.6, 0.7, 0.8):
        rec[f'hart_cons_{int(f*100)}'] = float(np.mean(hart_consensus_determinate(A, frac=f)))
    rows.append(rec)
df2 = pd.DataFrame(rows); df2.to_csv(f'{OUTDIR}/exp2_hart.csv', index=False)

berman = df2.berman.mean()
summary['exp2'] = dict(berman_det=float(berman), margin_hier=float((df2.berman - df2.hart_hier).mean()))
for f in (0.5, 0.6, 0.7, 0.8):
    summary['exp2'][f'margin_cons_{int(f*100)}'] = float((df2.berman - df2[f'hart_cons_{int(f*100)}']).mean())

print(f'Berman determinacy = {berman:.3f}')
print(f'margin over lexical-priority Hart = {(df2.berman - df2.hart_hier).mean():+.3f}')
for f in (0.5, 0.6, 0.7, 0.8):
    print(f'  margin over consensus Hart {int(f*100)}% = {(df2.berman - df2[f"hart_cons_{int(f*100)}"]).mean():+.3f}')

Berman determinacy = 0.862
margin over lexical-priority Hart = +0.272
  margin over consensus Hart 50% = -0.021
  margin over consensus Hart 60% = +0.270
  margin over consensus Hart 70% = +0.589
  margin over consensus Hart 80% = +0.786


*Reading.* Berman delivers more determinate law than any Hart that requires real
agreement (60 percent and up), and the advantage grows as Hart's requirement
tightens. A bare-majority Hart (50 percent) edges it out, but a model that finds
law whenever a slim majority leans one way has given up Hart's own point that
"where there is no consensus there is no law." The single headline number in the
earlier draft (+19 points) was one point on this curve, not a fixed fact.

## Conflicting principles: resolution without contradiction

Berman's second claim has two parts.

**Part 1: conflict lowers determinacy without destroying it.** We build four
kinds of case and report the determinacy rate of each: *aligned* (every principle
points the same way), *opposed* (four principles point strongly one way and four
strongly the other), *balanced* (a similar split but with weak activations), and
*random*. We expect aligned cases to be fully determinate and opposed cases to be
the hardest.

In [20]:
def make_cases(kind, n, rng):
    """Build n synthetic 9-principle cases of a given conflict type."""
    A = np.zeros((n, 9))
    if kind == 'aligned':                 # all principles point the same way, strongly
        A = np.abs(rng.uniform(0.3, 1.0, (n, 9)))
    elif kind == 'opposed':               # symmetric: 4 strong for, 4 strong against, 1 inactive
        A[:, :4] = rng.uniform(0.5, 1.0, (n, 4))
        A[:, 5:] = -rng.uniform(0.5, 1.0, (n, 4))
    elif kind == 'balanced':              # similar split but weak activations -> near cancellation
        A[:, :4] = rng.uniform(0.1, 0.5, (n, 4))
        A[:, 4:] = -rng.uniform(0.1, 0.5, (n, 5))
    else:                                  # random
        A = rng.uniform(-1, 1, (n, 9))
    return A

rows = []
for s in range(SEEDS):
    w = named_weight_vector('moderate', seed=s)
    rng = np.random.default_rng(2500 + s)
    rec = dict(seed=s)
    for k in ('aligned', 'opposed', 'balanced', 'random'):
        rec[k] = determinacy_rate(w, make_cases(k, 500, rng), 0.2)
    rows.append(rec)
df3 = pd.DataFrame(rows); df3.to_csv(f'{OUTDIR}/exp3_conflict.csv', index=False)
summary['exp3'] = {k: float(df3[k].mean()) for k in ('aligned', 'opposed', 'balanced', 'random')}
print({k: round(df3[k].mean(), 3) for k in ('aligned', 'opposed', 'balanced', 'random')})

{'aligned': np.float64(1.0), 'opposed': np.float64(0.519), 'balanced': np.float64(0.559), 'random': np.float64(0.864)}


**Part 2: aggregation cannot contradict itself.** This is the sharper point
against Hart. Suppose principles split into two camps. A validation model can
find that one camp is large enough to validate the outcome *and* the other camp
is large enough to validate its opposite. Then the same case is at once required
and forbidden: a contradiction. Or it can find neither camp large enough, leaving
a gap. A weighted sum has a single sign, so it always returns exactly one verdict
and can never contradict itself.

We make this concrete on the opposed cases. `hart_validation_status` marks each
case as *for*, *against*, *contradiction* (both camps validate), or *gap* (neither
does), using a simple rule: a direction is validated when at least two principles
are strongly activated that way. We then run Berman's model on the same cases and
confirm it produces no contradictions.

In [21]:
# F1's left panel. Averaged over the same thirty seeds the rest of this notebook
# uses. An earlier version measured seed 0 alone, which turns out to be the single
# most favourable of the thirty (0.792 against a 30-seed mean of 0.525); the per-seed
# spread is stored in the summary so the figure can never silently report one draw.
hart_acc, berm_acc, per_seed = [], [], []
for s in range(SEEDS):
    w     = named_weight_vector('moderate', seed=s)
    rng   = np.random.default_rng(2600 + s)
    A_opp = make_cases('opposed', 6000, rng)          # the hard, two-camp cases

    # Naive validation foil: a direction is 'valid law' if >= 2 principles strongly
    # (|a| > 0.5) point that way. Both camps can clear the bar at once.
    codes = hart_validation_status(A_opp, k=2, act_cut=0.5)
    hart_acc.append({'for': float(np.mean(codes == 1)), 'against': float(np.mean(codes == 2)),
                     'contradiction': float(np.mean(codes == 3)), 'gap': float(np.mean(codes == 0))})

    # Berman's model on the same cases: one signed net force per case.
    F = net_force(w, A_opp)
    berm_acc.append({'determinate_for': float(np.mean(F > 0.2)),
                     'determinate_against': float(np.mean(F < -0.2)),
                     'indeterminate': float(np.mean(np.abs(F) <= 0.2)),
                     'contradiction': 0.0})           # impossible: one sign cannot be both
    per_seed.append(berm_acc[-1]['determinate_for'] + berm_acc[-1]['determinate_against'])

hart   = {k: float(np.mean([h[k] for h in hart_acc])) for k in hart_acc[0]}
berman = {k: float(np.mean([b[k] for b in berm_acc])) for k in berm_acc[0]}
summary['exp3b'] = dict(hart=hart, berman=berman, n_seeds=int(SEEDS),
                        single_verdict_by_seed=[round(v, 4) for v in per_seed])

sv = berman['determinate_for'] + berman['determinate_against']
print(f'Naive validation on opposed cases ({SEEDS}-seed means):')
print(f"  contradiction {hart['contradiction']:.1%} | gap {hart['gap']:.1%} | "
      f"single verdict {hart['for']+hart['against']:.1%}")
print('Berman aggregation on the same cases:')
print(f"  contradiction {berman['contradiction']:.1%} | indeterminate {berman['indeterminate']:.1%} | "
      f"single verdict {sv:.1%}")
print(f'  per-seed single verdict: min {min(per_seed):.3f} | median {np.median(per_seed):.3f} | max {max(per_seed):.3f}')
print('-> the validation rule contradicts itself on every conflict case; the sum never does.')
assert abs(hart['contradiction'] - 1.0) < 1e-9, 'validation foil must contradict on all opposed cases'
assert berman['contradiction'] == 0.0, 'weighted sum can never contradict itself'


Naive validation on opposed cases (30-seed means):
  contradiction 100.0% | gap 0.0% | single verdict 0.0%
Berman aggregation on the same cases:
  contradiction 0.0% | indeterminate 47.6% | single verdict 52.4%
  per-seed single verdict: min 0.441 | median 0.493 | max 0.792
-> the validation rule contradicts itself on every conflict case; the sum never does.


## R3 — Determination is nonlexical

This is the claim that separates Berman from both Hart and a lexical (priority)
model. In a lexical model the outcome turns on which single principle ranks
highest; the exact magnitudes do not matter. In Berman's model the magnitudes are
added, so they do matter.

We test this directly. Take a case, then change the activation magnitudes while
holding every sign and every rank order fixed (the largest magnitude stays
largest, and so on). A lexical rule cannot change its answer under this
perturbation, because the top-ranked principle is unchanged. A summing rule can,
because the total moved. `nonlexicality_score` reports how often each rule's
verdict changes. For the sum we report two figures: changes of *direction*
(measured with the threshold at zero) and changes including the determinacy
boundary (threshold 0.2). A clean separation — the sum responds, the lexical rule
does not — is the signature of nonlexical determination.

The same cell checks the "rough weights" point from the other side. We round the
weights to five coarse bins, matching Berman's vocabulary of slight, moderate,
weighty, very weighty, and nearly conclusive, and confirm the coarse weights
tolerate about as much noise as fine-grained ones.

In [22]:
def coarse5(w):
    """Round each weight to one of Berman's five rough bins."""
    centers = np.array([0.1, 0.3, 0.5, 0.7, 0.9])      # slight ... nearly conclusive
    return centers[np.digitize(w, [0.2, 0.4, 0.6, 0.8])]

rows = []
for s in range(SEEDS):
    wm = named_weight_vector('moderate', seed=s)
    wh = named_weight_vector('high', seed=s)           # high-variance weights span all five bins
    A  = random_activations(1000, 9, np.random.default_rng(3000 + s))
    rows.append(dict(
        seed=s,
        # nonlexicality: does the verdict respond to magnitude-only perturbation?
        nl_saw_dir =nonlexicality_score(lambda X: saw_verdict(wm, X, tau=0.0), A, seed=s),  # direction changes
        nl_saw_full=nonlexicality_score(lambda X: saw_verdict(wm, X, tau=0.2), A, seed=s),  # + boundary changes
        nl_lex     =nonlexicality_score(lambda X: lexicographic_verdict(wm, X), A, seed=s), # lexical foil
        # rough weights: fine-grained vs five-bin tolerance
        nbins=int(len(np.unique(coarse5(wh)))),
        sig_fine   =critical_sigma(wh, A, seed=s),
        sig_coarse5=critical_sigma(coarse5(wh), A, seed=s)))
df4 = pd.DataFrame(rows); df4.to_csv(f'{OUTDIR}/exp4_nonlexicality.csv', index=False)
summary['exp4'] = {c: float(df4[c].mean()) for c in df4.columns if c != 'seed'}
summary['exp4']['nonlex_ratio_dir'] = float(df4.nl_saw_dir.mean() / max(df4.nl_lex.mean(), 1e-9))

print('Nonlexicality (verdict changes under magnitude-only perturbation):')
print(f'  sum, direction (tau=0)      = {df4.nl_saw_dir.mean():.4f}')
print(f'  sum, incl. boundary (tau=.2)= {df4.nl_saw_full.mean():.4f}')
print(f'  lexical foil                = {df4.nl_lex.mean():.5f}')
print(f'  -> the sum is {df4.nl_saw_dir.mean()/max(df4.nl_lex.mean(),1e-9):.0f}x more magnitude-sensitive than the lexical rule')
print(f'Rough weights: fine sigma = {df4.sig_fine.mean():.3f} vs coarse {int(df4.nbins.median())}-bin sigma = {df4.sig_coarse5.mean():.3f}  (about equal)')

Nonlexicality (verdict changes under magnitude-only perturbation):
  sum, direction (tau=0)      = 0.0256
  sum, incl. boundary (tau=.2)= 0.0502
  lexical foil                = 0.00263
  -> the sum is 10x more magnitude-sensitive than the lexical rule
Rough weights: fine sigma = 0.068 vs coarse 4-bin sigma = 0.069  (about equal)


## D2 — Correlated activations (descriptive)

The experiments above draw each principle's activation independently. Real cases
are not like that: textual arguments tend to move together, and moral arguments
tend to move together. Independence is therefore the hard case, and the numbers
above are conservative. Here we re-measure the weight tolerance under three
activation structures to show the direction of the effect. This is reported for
context. Berman's theory does not require the model to benefit from correlation,
so we do not treat it as a test of the theory.

In [23]:
rows = []
for s in range(SEEDS):
    w = named_weight_vector('moderate', seed=s)
    rng = np.random.default_rng(4000 + s)
    rows.append(dict(
        seed=s,
        sig_independent=critical_sigma(w, random_activations(800, 9, rng), seed=s),          # the baseline above
        sig_category   =critical_sigma(w, category_correlated_activations(800, rng), seed=s), # within-category correlation
        sig_full       =critical_sigma(w, fully_correlated_activations(800, 9, rng), seed=s)))# all principles correlated
df5 = pd.DataFrame(rows); df5.to_csv(f'{OUTDIR}/exp5_correlation.csv', index=False)
summary['exp5'] = {c: float(df5[c].mean()) for c in df5.columns if c != 'seed'}
summary['exp5']['full_censored'] = int((df5.sig_full >= CRITICAL_SIGMA_CEIL - 1e-9).sum())
print({c: round(df5[c].mean(), 3) for c in df5.columns if c != 'seed'})
print(f'fully-correlated tolerance is right-censored in {int((df5.sig_full >= CRITICAL_SIGMA_CEIL-1e-9).sum())}/{SEEDS} '
      f'seeds (the true value is >= {CRITICAL_SIGMA_CEIL}); category correlation roughly doubles the independent baseline')

{'sig_independent': np.float64(0.069), 'sig_category': np.float64(0.156), 'sig_full': np.float64(1.0)}
fully-correlated tolerance is right-censored in 30/30 seeds (the true value is >= 1.0); category correlation roughly doubles the independent baseline


## Two worked cases from Berman's paper

To connect the model to real disputes we reproduce the two cases Berman works
through in the article.

**TVA v. Hill (the snail darter case).** Three principles bear on whether the
Endangered Species Act stopped the nearly finished dam: the plain meaning of the
text, the enacting Congress's intent, and avoiding an absurd result. On Berman's
reading, Burger (majority) and Powell (dissent) agree on which principles matter
and roughly how weighty each is; they disagree about how strongly each applies.
We set the weights once and change only the activations between the two Justices.

**Obergefell.** Equal dignity and the pursuit of happiness point toward a right to
same-sex marriage; textual meaning, the enactors' intent, precedent, and
majoritarianism point the other way more weakly. The net force points toward the
right. These values are illustrative and follow Berman's own diagram.

In [24]:
# TVA v. Hill: one fixed weight vector, two readings of the activations.
tva_w  = np.array([0.85, 0.70, 0.60])     # text, congressional intent, avoid absurdity
burger = np.array([0.9, 0.8, -0.2])       # text and intent strong; absurdity weak
# Congressional intent is 0 for Powell, not weakly positive. Berman draws it in
# Figure 9 as a DOUBLE-HEADED arrow: the principle is engaged but points both ways,
# which is his rendering of "there was no actual congressional intention relevant to
# this dispute -- no intention either that completion ... would be illegal or that it
# would not be" (HPMP 340). In a signed scalar model that is a net contribution of
# zero. Note this is NOT the same state as a principle that is simply not engaged;
# the scalar model cannot tell the two apart, which the manuscript flags.
powell = np.array([0.3, 0.0, -0.9])       # text weak; intent balanced to zero; absurdity strong
print(f'TVA v. Hill   Burger F = {net_force(tva_w, burger):+.3f}   Powell F = {net_force(tva_w, powell):+.3f}   (tau = 0.2)')

# Obergefell: positive activation = toward a right to same-sex marriage.
ob_w = np.array([0.80, 0.65, 0.85, 0.70, 0.75, 0.55])  # dignity, happiness, text, intent, precedent, majoritarianism
ob_a = np.array([+0.9, +0.7, -0.2, -0.3, -0.3, -0.5])
F_ob = net_force(ob_w, ob_a)
print(f'Obergefell    F = {F_ob:+.3f}  -> {"determinate for the right" if F_ob > 0.2 else "indeterminate"}')

pd.DataFrame([dict(case='TVA_Burger', F=net_force(tva_w, burger)),
              dict(case='TVA_Powell', F=net_force(tva_w, powell)),
              dict(case='Obergefell', F=F_ob)]).to_csv(f'{OUTDIR}/exp6_worked.csv', index=False)
summary['worked'] = dict(tva_burger=float(net_force(tva_w, burger)),
                         tva_powell=float(net_force(tva_w, powell)), obergefell=float(F_ob))

TVA v. Hill   Burger F = +1.205   Powell F = -0.285   (tau = 0.2)
Obergefell    F = +0.295  -> determinate for the right


## Write the summary
All headline numbers are collected into one JSON file for the figures notebook
and for the self-check below.

In [25]:
json.dump(summary, open(f'{OUTDIR}/notebook0_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))

{
  "exp1": {
    "mean_sigma": 0.09006333717342282,
    "sigma_min": 0.07068464673717868,
    "sigma_max": 0.13434179034473148,
    "n_censored": 0,
    "n_configs": 45,
    "spread_tau": 0.048972364438744906,
    "spread_count": 0.00784141633715485,
    "spread_dist": 0.0011779882974415762
  },
  "exp2": {
    "berman_det": 0.8619000000000001,
    "margin_hier": 0.27176666666666666,
    "margin_cons_50": -0.021100000000000018,
    "margin_cons_60": 0.2699,
    "margin_cons_70": 0.5885666666666666,
    "margin_cons_80": 0.7861999999999999
  },
  "exp3": {
    "aligned": 1.0,
    "opposed": 0.5185333333333333,
    "balanced": 0.5590666666666666,
    "random": 0.8642666666666667
  },
  "exp3b": {
    "hart": {
      "for": 0.0,
      "against": 0.0,
      "contradiction": 1.0,
      "gap": 0.0
    },
    "berman": {
      "determinate_for": 0.21532222222222225,
      "determinate_against": 0.30843333333333334,
      "indeterminate": 0.4762444444444444,
      "contradiction": 0.0
    },


## Self-check

This cell re-derives the soundness properties and compares the headline numbers
against the committed reference values. With fixed seeds a fresh run reproduces
these, so the comparison uses a small tolerance only to absorb library
differences. If anything fails the cell raises an error.

## E2c: the Hart comparison is distribution-dependent

R4 is scored on independent activations, but the mechanisms of Part VI and the divergence battery run
on category-correlated ones. This cell reports the comparison on both and diagnoses the difference:
under correlation a lexical rule reaches consensus in its top tier almost always, so it decides most
cases on the two textual principles without ever consulting the other seven.

In [26]:
# --- E2c: the Hart comparison is distribution-dependent, and here is why ---
# R4 is scored on independent activations. The mechanisms and the divergence battery run on
# category-correlated ones. This cell reports the comparison on BOTH, and diagnoses the difference.
_np2 = len(NAMED_PRINCIPLES)
_tier = np.zeros(_np2, int)
for _c, _t in (('textual', 0), ('structural', 1), ('moral', 2)):
    for _i in CATEGORY_INDEX[_c]: _tier[_i] = _t
_w2c = named_weight_vector('moderate', seed=0)
rows = []
for lab, A2 in (('independent', random_activations(40000, _np2, np.random.default_rng(11))),
                ('category-correlated', category_correlated_activations(40000, np.random.default_rng(11), rho=0.7))):
    hart = float(hart_hierarchical_determinate(_w2c, A2, _tier).mean())
    saw  = float(determinacy_rate(_w2c, A2, tau=DEFAULT_TAU))
    rows.append(dict(activations=lab, hart=hart, saw=saw, margin=saw - hart))
    print(f"  {lab:20s} Hart {hart:.4f}   weighted sum {saw:.4f}   margin {saw-hart:+.4f}")

# why: which tier is dispositive? A lexical rule stops at the highest tier with any active principle.
A2 = category_correlated_activations(40000, np.random.default_rng(11), rho=0.7)
_h = hart_hierarchical_determinate(_w2c, A2, _tier)
_res = np.full(len(A2), -1)
for _t in (0, 1, 2):
    _sub = A2[:, np.where(_tier == _t)[0]]
    _act = (np.sum(_sub > 0.1, 1) + np.sum(_sub < -0.1, 1)) > 0
    _first = (_res == -1) & _act; _res[_first] = _t
disp = {int(t): float((_res == t).mean()) for t in (0, 1, 2)}
print(f"  under correlation the dispositive tier is: textual {disp[0]:.3f}, "
      f"structural {disp[1]:.3f}, moral {disp[2]:.3f}")
print('  -> lexical Hart decides most correlated cases on the two textual principles alone,')
print('     never reaching the other seven. Its determinacy is textual-only determinacy.')
pd.DataFrame(rows).to_csv(f'{OUTDIR}/exp2c_hart_by_distribution.csv', index=False)
summary['exp2c'] = dict(by_distribution={r['activations']: r for r in rows}, dispositive_tier=disp)


  independent          Hart 0.5925   weighted sum 0.8605   margin +0.2680
  category-correlated  Hart 0.9558   weighted sum 0.9015   margin -0.0543
  under correlation the dispositive tier is: textual 0.957, structural 0.042, moral 0.001
  -> lexical Hart decides most correlated cases on the two textual principles alone,
     never reaching the other seven. Its determinacy is textual-only determinacy.


## E2d: how much of the law does each rule bring to bear?

E2c found a lexical rule settling more correlated cases than the weighted sum. That looks like a
point against Berman until one asks what the determinacy costs. A rule with strict priority stops at
the highest tier that has anything active in it and never reaches the tiers below, however strongly
they apply. This cell measures how many principles each rule hears and how much of the force in play
it sets aside.

In [27]:
# --- E2d: how much of the law does each rule actually bring to bear? ---
# E2c showed a lexical rule settling MORE correlated cases than the weighted sum. This cell asks what
# that determinacy costs. Two measures, same cases, same activity cutoff:
#   CONSULTED     : how many active principles the rule is capable of hearing.
#   FORCE IGNORED : the share of the case's total weighted force the rule never considers.
_np2d = len(NAMED_PRINCIPLES)
_tier2d = np.zeros(_np2d, int)
for _c, _t in (('textual', 0), ('structural', 1), ('moral', 2)):
    for _i in CATEGORY_INDEX[_c]: _tier2d[_i] = _t
_w2d, _CUT = named_weight_vector('moderate', seed=0), 0.1

rows2d = []
for lab, A in (('independent', random_activations(20000, _np2d, np.random.default_rng(31))),
               ('correlated', category_correlated_activations(20000, np.random.default_rng(31), rho=0.7))):
    active = np.abs(A) > _CUT
    mass = np.abs(A) * _w2d[None, :]                      # weighted force each principle carries
    total = mass.sum(1)

    # the dispositive tier: the highest-priority tier with any active principle. A lexical rule
    # stops there and never reaches the tiers below, however strongly they apply.
    res = np.full(len(A), -1)
    for t in np.unique(_tier2d):
        sub = A[:, np.where(_tier2d == t)[0]]
        act = (np.sum(sub > _CUT, 1) + np.sum(sub < -_CUT, 1)) > 0
        first = (res == -1) & act; res[first] = t
    heard = np.zeros_like(A, dtype=bool)
    for t in np.unique(_tier2d):
        heard[res == t] = (_tier2d == t)                   # only the dispositive tier is heard

    saw_consult  = float(active.sum(1).mean())
    hart_consult = float((active & heard).sum(1).mean())
    saw_ignored  = 0.0                                     # the sum reads every principle
    hart_ignored = float(np.mean(1.0 - (mass * heard).sum(1) / np.where(total == 0, 1, total)))
    rows2d.append(dict(activations=lab, saw_consulted=saw_consult, hart_consulted=hart_consult,
                       saw_force_ignored=saw_ignored, hart_force_ignored=hart_ignored))
    print(f"  {lab:12s} consulted: weighted sum {saw_consult:.2f} of {_np2d}, lexical Hart {hart_consult:.2f}"
          f"   |  force ignored: {saw_ignored:.0%} vs {hart_ignored:.0%}")

pd.DataFrame(rows2d).to_csv(f'{OUTDIR}/exp2d_principles_brought_to_bear.csv', index=False)
summary['exp2d'] = {r['activations']: r for r in rows2d}
_c2d = summary['exp2d']['correlated']
print(f"\n  Under correlation a lexical rule settles more cases than the weighted sum (E2c), but it")
print(f"  hears {_c2d['hart_consulted']:.2f} of {_np2d} principles and sets aside "
      f"{_c2d['hart_force_ignored']:.0%} of the force in play.")
print( "  Determinacy counted on its own rewards foreclosure: a rule can settle more by listening less.")


  independent  consulted: weighted sum 8.10 of 9, lexical Hart 1.83   |  force ignored: 0% vs 78%
  correlated   consulted: weighted sum 7.72 of 9, lexical Hart 1.83   |  force ignored: 0% vs 76%

  Under correlation a lexical rule settles more cases than the weighted sum (E2c), but it
  hears 1.83 of 9 principles and sets aside 76% of the force in play.
  Determinacy counted on its own rewards foreclosure: a rule can settle more by listening less.


In [28]:
ok = True
def check(name, cond):
    global ok; ok = ok and bool(cond); print(('PASS' if cond else 'FAIL'), '-', name)

_w = named_weight_vector('moderate', 0)
_A = random_activations(2000, 9, np.random.default_rng(0))

# The nonlexicality metric must give exactly zero for a strictly lexical rule
# (decide by the single largest-magnitude principle). If it does not, the metric
# is leaking and the R3 claim is unreliable.
def _topmag(X):
    X = np.atleast_2d(X); j = np.argmax(np.abs(X), axis=1)
    return np.sign(X[np.arange(len(X)), j]).astype(int)
check('R3 metric: pure lexical rule scores exactly 0', nonlexicality_score(_topmag, _A, seed=1) == 0.0)
check('R3: the sum is >5x more magnitude-sensitive than the lexical foil', summary['exp4']['nonlex_ratio_dir'] > 5)

# A single net force has a single sign, so Berman never contradicts itself.
check('no contradiction under aggregation', summary['exp3b']['berman']['contradiction'] == 0.0)
check('validation does contradict on opposed cases', summary['exp3b']['hart']['contradiction'] > 0.5)

# The verdict and determinacy must use the same strict |F|>tau rule.
check('verdict and determinacy agree', float(np.mean(saw_verdict(_w, _A) != 0)) == determinacy_rate(_w, _A))
check('net_force([.5,.5],[1,-1]) == 0', abs(net_force(np.array([.5, .5]), np.array([1., -1.]))) < 1e-12)
check('bootstrap is reproducible', bootstrap_ci(_A[:, 0], seed=3) == bootstrap_ci(_A[:, 0], seed=3))
check('no censored tolerance configs in exp1', summary['exp1']['n_censored'] == 0)

# Headline values, within tolerance of the committed reference.
REF = dict(mean_sigma=0.0901, margin_hier=0.2718, sig_category=0.1558,
           tva_burger=1.205, tva_powell=-0.285)
check('R2 mean sigma ~ 0.090', abs(summary['exp1']['mean_sigma'] - REF['mean_sigma']) < 5e-3)
check('R4 margin over lexical Hart ~ +0.27', abs(summary['exp2']['margin_hier'] - REF['margin_hier']) < 5e-3)
check('D2 category tolerance ~ 0.156', abs(summary['exp5']['sig_category'] - REF['sig_category']) < 5e-3)
check('worked: TVA Burger == +1.205', abs(summary['worked']['tva_burger'] - REF['tva_burger']) < 1e-6)
check('worked: TVA Powell == -0.285', abs(summary['worked']['tva_powell'] - REF['tva_powell']) < 1e-6)


# E2c: the Hart comparison across activation distributions
_b = summary['exp2c']['by_distribution']
check('E2c margin is positive on independent activations', _b['independent']['margin'] > 0.2)
check('E2c margin is NEGATIVE on correlated activations', _b['category-correlated']['margin'] < 0)
check('E2c Hart decides far more under correlation', _b['category-correlated']['hart'] > _b['independent']['hart'] + 0.3)
check('E2c the top tier is dispositive on >90% of correlated cases', summary['exp2c']['dispositive_tier'][0] > 0.90)


# E2d: what the lexical rule's determinacy costs
_d2 = summary['exp2d']['correlated']
check('E2d the weighted sum hears nearly every active principle', _d2['saw_consulted'] > 7.0)
check('E2d the lexical rule hears fewer than two', _d2['hart_consulted'] < 2.0)
check('E2d the weighted sum sets aside no force', _d2['saw_force_ignored'] == 0.0)
check('E2d the lexical rule sets aside most of it (>0.6)', _d2['hart_force_ignored'] > 0.6)
check('E2d the gap holds on independent activations too', summary['exp2d']['independent']['hart_force_ignored'] > 0.6)

print('\n' + ('ALL CHECKS PASSED' if ok else 'SOME CHECKS FAILED'))
summary['self_check_passed'] = bool(ok)
assert ok, 'self-check failed'

# the summary is written earlier in this notebook, before the experiments added later;
# re-write it here so every experiment and the self-check flag are captured.
import json; json.dump(summary, open(f'{OUTDIR}/notebook0_summary.json', 'w'), indent=2, default=float)


PASS - R3 metric: pure lexical rule scores exactly 0
PASS - R3: the sum is >5x more magnitude-sensitive than the lexical foil
PASS - no contradiction under aggregation
PASS - validation does contradict on opposed cases
PASS - verdict and determinacy agree
PASS - net_force([.5,.5],[1,-1]) == 0
PASS - bootstrap is reproducible
PASS - no censored tolerance configs in exp1
PASS - R2 mean sigma ~ 0.090
PASS - R4 margin over lexical Hart ~ +0.27
PASS - D2 category tolerance ~ 0.156
PASS - worked: TVA Burger == +1.205
PASS - worked: TVA Powell == -0.285
PASS - E2c margin is positive on independent activations
PASS - E2c margin is NEGATIVE on correlated activations
PASS - E2c Hart decides far more under correlation
PASS - E2c the top tier is dispositive on >90% of correlated cases
PASS - E2d the weighted sum hears nearly every active principle
PASS - E2d the lexical rule hears fewer than two
PASS - E2d the weighted sum sets aside no force
PASS - E2d the lexical rule sets aside most of it (>0.6

## What this notebook establishes

With weights stipulated by hand, the aggregation model behaves as Berman
describes. It tolerates rough weights (R2). Its verdicts depend on activation
magnitudes, not just on which principle ranks highest (R3). It produces more
determinate law than any Hart-style model that requires real consensus (R4), and
it never contradicts itself the way a validation model can. Correlated
activations, the realistic case, only help (D2).

This is the formal layer only. It shows the machinery works **when handed good
weights**. It does not show that the practice of judging produces such weights.
That is the harder question, and it is where the theory is actually on the hook.
The next notebooks take it up: Notebook 1 asks whether weights emerge from a
population of judges at all, and Notebook 1c implements the mechanism Berman
actually describes, in which a principle gains weight by being taken up and relied
upon in decisions that remain good law.